# 🏗️ Data Generation — Run 2 (Pipeline Test)
> **01B** — Identical structure to `01_data_generation.ipynb`. Different seed, slightly larger dataset.
> Run this → hit **Refresh** in Power BI → dashboard updates automatically.
> All column names, table names, and view names are identical to Run 1.

## 0 · Imports & MySQL Connection

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()
np.random.seed(99)  # Different seed from Run 1 (was 42)

DATA_DIR    = "../data/generated_data"
TABLEAU_DIR = "../data/tableau"
FIGURES_DIR = "../figures"
SQL_DIR     = "../data/sql"

for d in [DATA_DIR, TABLEAU_DIR, FIGURES_DIR, SQL_DIR]:
    os.makedirs(d, exist_ok=True)

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}",
    echo=False
)

# Drop all tables before regenerating (clean slate)
with engine.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
    for t in ["return_conditions", "inventory_events", "rentals",
              "rental_revenue_vs_discount", "customers", "pricing_rules",
              "products", "categories"]:
        conn.execute(text(f"DROP TABLE IF EXISTS `{t}`"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))

def save(name, df):
    """Write DataFrame to MySQL and CSV simultaneously."""
    df.to_sql(name, engine, if_exists="replace", index=False)
    df.to_csv(f"{DATA_DIR}/{name}.csv", index=False)
    with engine.connect() as conn:
        n = conn.execute(text(f"SELECT COUNT(*) FROM `{name}`")).scalar()
    print(f"  {name}: {n:,} rows")

print(f"Connected: {os.getenv('DB_NAME')}")
print("Run 2 ready — seed=99, ~600 products, date range extended to 2025-03-31")


Connected: rental_final_project
Run 2 ready — seed=99, ~600 products, date range extended to 2025-03-31


## 1 · Categories

In [3]:
categories_data = [
    {"category_id": 1,  "category_name": "Smartphones",  "depreciation_class": "fast",     "avg_depreciation_rate": 0.22, "rental_demand_tier": "high"},
    {"category_id": 2,  "category_name": "Laptops",      "depreciation_class": "fast",     "avg_depreciation_rate": 0.18, "rental_demand_tier": "high"},
    {"category_id": 3,  "category_name": "Tablets",      "depreciation_class": "fast",     "avg_depreciation_rate": 0.16, "rental_demand_tier": "medium"},
    {"category_id": 4,  "category_name": "Wearables",    "depreciation_class": "fast",     "avg_depreciation_rate": 0.20, "rental_demand_tier": "medium"},
    {"category_id": 5,  "category_name": "Drones",       "depreciation_class": "standard", "avg_depreciation_rate": 0.14, "rental_demand_tier": "medium"},
    {"category_id": 6,  "category_name": "Audio",        "depreciation_class": "standard", "avg_depreciation_rate": 0.10, "rental_demand_tier": "medium"},
    {"category_id": 7,  "category_name": "Gaming",       "depreciation_class": "standard", "avg_depreciation_rate": 0.12, "rental_demand_tier": "high"},
    {"category_id": 8,  "category_name": "Cameras",      "depreciation_class": "standard", "avg_depreciation_rate": 0.11, "rental_demand_tier": "medium"},
    {"category_id": 9,  "category_name": "TVs",          "depreciation_class": "slow",     "avg_depreciation_rate": 0.07, "rental_demand_tier": "low"},
    {"category_id": 10, "category_name": "Appliances",   "depreciation_class": "slow",     "avg_depreciation_rate": 0.06, "rental_demand_tier": "low"},
    {"category_id": 11, "category_name": "Keyboards",    "depreciation_class": "slow",     "avg_depreciation_rate": 0.05, "rental_demand_tier": "low"},
    {"category_id": 12, "category_name": "Monitors",     "depreciation_class": "slow",     "avg_depreciation_rate": 0.08, "rental_demand_tier": "low"},
    {"category_id": 13, "category_name": "Networking",   "depreciation_class": "standard", "avg_depreciation_rate": 0.09, "rental_demand_tier": "medium"},
    {"category_id": 14, "category_name": "Peripherals",  "depreciation_class": "slow",     "avg_depreciation_rate": 0.05, "rental_demand_tier": "low"},
]
categories = pd.DataFrame(categories_data)
save("categories", categories)


  categories: 14 rows


## 2 · Pricing Rules

In [4]:
pricing_data = []
rule_id = 1
for pricing_model in ["flat_rate", "pct_of_retail"]:
    for duration_model in ["7_day", "30_day", "flexible"]:
        for experiment_group in ["A", "B"]:
            pricing_data.append({
                "rule_id":              rule_id,
                "pricing_model":        pricing_model,
                "duration_model":       duration_model,
                "experiment_group":     experiment_group,
                "base_daily_rate":      round(np.random.uniform(3, 12), 2),
                "pct_of_retail_daily":  round(np.random.uniform(0.008, 0.014), 4),
                "min_rental_days":      1 if duration_model == "flexible" else (7 if duration_model == "7_day" else 30),
                "max_rental_days":      90,
                "late_fee_per_day":     round(np.random.uniform(2, 8), 2),
                "security_deposit_pct": round(np.random.uniform(0.10, 0.25), 2),
                "insurance_fee_pct":    round(np.random.uniform(0.02, 0.05), 3),
                "created_at":           "2021-01-01",
            })
            rule_id += 1
pricing = pd.DataFrame(pricing_data)
save("pricing_rules", pricing)


  pricing_rules: 12 rows


## 3 · Seasonal Demand Tables

In [5]:
SEASONAL_STD  = {1:0.75,2:0.75,3:0.85,4:0.90,5:0.90,6:0.85,7:0.80,8:0.85,9:0.95,10:1.05,11:1.20,12:1.35}
SEASONAL_HIGH = {1:0.70,2:0.70,3:0.80,4:0.85,5:0.85,6:0.80,7:0.75,8:0.80,9:0.90,10:1.10,11:1.40,12:1.60}
SEASONAL_LOW  = {1:0.90,2:0.90,3:0.95,4:1.00,5:1.00,6:0.95,7:0.90,8:0.90,9:1.00,10:1.05,11:1.05,12:1.10}
HIGH_SEASON_CATS = {1, 2, 7}  # Smartphones, Laptops, Gaming

def get_seasonal_table(cat_id, demand_tier):
    if cat_id in HIGH_SEASON_CATS:
        return SEASONAL_HIGH
    elif demand_tier == "low":
        return SEASONAL_LOW
    return SEASONAL_STD

print("Seasonal tables defined.")


Seasonal tables defined.


## 4 · Products

In [6]:
brands_by_cat = {
    1:["Apple","Samsung","Xiaomi","OnePlus"],  2:["Apple","Dell","HP","Lenovo","Asus"],
    3:["Apple","Samsung","Lenovo","Huawei"],   4:["Apple","Garmin","Fitbit","Samsung"],
    5:["DJI","Parrot","Autel"],                6:["Sony","Bose","JBL","Sennheiser"],
    7:["Sony","Microsoft","Nintendo","Razer"], 8:["Canon","Nikon","Sony","Fujifilm"],
    9:["Samsung","LG","Sony","Philips"],       10:["Dyson","Bosch","Philips","Tefal"],
    11:["Logitech","Corsair","Razer","Microsoft"], 12:["Dell","LG","Benq","AOC"],
    13:["TP-Link","Netgear","Asus","Cisco"],   14:["Logitech","Microsoft","Anker","Belkin"],
}
suffixes = ["Pro","Plus","Ultra","SE","X","Max","Lite",""]

# Run 2: extended date range + slightly more products per category
PROG_END  = datetime(2025, 3, 31)  # Extended vs Run 1 (was 2024-12-31)
n_per_cat = [52,57,40,36,30,43,40,43,36,32,40,36,32,47]  # ~564 total vs 500

def random_listed_date():
    year = np.random.choice([2021,2022,2023,2024], p=[0.10,0.25,0.40,0.25])
    if year == 2024:
        return datetime(2024,1,1) + timedelta(days=int(np.random.uniform(0,181)))
    return datetime(year,1,1) + timedelta(days=int(np.random.uniform(0,365)))

products_list = []
pid = 1
for cat in categories_data:
    cid = cat["category_id"]
    for _ in range(n_per_cat[cid-1]):
        retail = round(np.random.choice(
            [np.random.uniform(50,200), np.random.uniform(200,600), np.random.uniform(600,1800)],
            p=[0.3,0.4,0.3]), 2)
        listed = random_listed_date()
        elig   = listed + timedelta(days=300)
        yrs    = max(0, (PROG_END-listed).days/365)
        dep    = max(0.03, min(cat["avg_depreciation_rate"]+np.random.normal(0,0.02), 0.30))
        products_list.append({
            "product_id":                pid,
            "category_id":               cid,
            "product_name":              f"{np.random.choice(brands_by_cat[cid])} {cat['category_name'].rstrip('s')} {np.random.choice(suffixes)}".strip(),
            "brand":                     np.random.choice(brands_by_cat[cid]),
            "original_retail_price":     retail,
            "current_depreciated_value": round(retail*max(0.1,1-dep*yrs),2),
            "condition_grade":           np.random.choice(["A","B","C"],p=[0.5,0.35,0.15]),
            "listed_date":               listed.date(),
            "rental_eligible_date":      elig.date(),
            "retailer":                  np.random.choice(["FNAC Portugal","Amazon ES"],p=[0.55,0.45]),
            "is_active":                 1,
        })
        pid += 1

products = pd.DataFrame(products_list)
save("products", products)


  products: 564 rows


## 5 · Customers

In [7]:
first_names = ["Ana","Pedro","Maria","João","Sofia","Miguel","Inês","Ricardo","Beatriz","Tiago",
               "Carlos","Luísa","Fernando","Catarina","André","Marta","Rui","Sara","Diogo","Filipa",
               "Elena","Marco","Lucia","Pablo","Rosa","Diego","Carmen","Rafael","Isabel","Nuno"]
last_names  = ["Silva","Santos","Ferreira","Pereira","Costa","Oliveira","Rodrigues","Martins",
               "Jesus","Sousa","Fernández","García","López","Martínez","González","Sánchez"]
cities_pt   = ["Lisboa","Porto","Braga","Coimbra","Setúbal","Faro","Évora","Aveiro","Funchal","Leiria"]
cities_es   = ["Madrid","Barcelona","Valencia","Sevilla","Zaragoza","Málaga","Bilbao","Alicante"]
segments    = ["student","professional","business","casual"]
seg_w       = [0.25,0.35,0.15,0.25]

customers_list = []
for cid in range(1, 2201):  # 2,200 customers vs 2,000 in Run 1
    country = np.random.choice(["PT","ES"], p=[0.6,0.4])
    reg = datetime(2021,1,1) + timedelta(days=int(np.random.uniform(0,365*2)))
    customers_list.append({
        "customer_id":       cid,
        "first_name":        np.random.choice(first_names),
        "last_name":         np.random.choice(last_names),
        "city":              np.random.choice(cities_pt if country=="PT" else cities_es),
        "country":           country,
        "customer_segment":  np.random.choice(segments, p=seg_w),
        "registration_date": reg.date(),
    })
customers = pd.DataFrame(customers_list)
save("customers", customers)


  customers: 2,200 rows


## 6 · Customer Repeat Rental Pool

In [8]:
SEG_RENTAL_DIST = {
    "business":     ([3,4,5,6,7,8], [0.10,0.20,0.25,0.22,0.15,0.08]),
    "professional": ([2,3,4,5],     [0.25,0.35,0.25,0.15]),
    "student":      ([1,2,3],       [0.45,0.38,0.17]),
    "casual":       ([1,2,3],       [0.55,0.33,0.12]),
}
SEG_MONTH_BOOST = {
    "student":      {1:1.40,2:1.10,9:1.60,10:1.20},
    "casual":       {6:1.20,7:1.25,8:1.20,11:1.30,12:1.50},
    "professional": {},
    "business":     {},
}
customer_pool = []
for _, row in customers.iterrows():
    vals, probs = SEG_RENTAL_DIST[row["customer_segment"]]
    n = np.random.choice(vals, p=probs)
    customer_pool.extend([row["customer_id"]] * n)
np.random.shuffle(customer_pool)
pool_idx = [0]

def next_customer(month=None):
    cid = customer_pool[pool_idx[0] % len(customer_pool)]
    pool_idx[0] += 1
    return cid

print(f"Customer pool: {len(customer_pool):,} slots")


Customer pool: 6,236 slots


## 7 · Rentals, Returns & Inventory Events

In [9]:
eligible = products[pd.to_datetime(products["rental_eligible_date"]) <= pd.Timestamp(PROG_END)].copy()
rentals_list, returns_list, events_list = [], [], []
rid = 1

for _, prod in eligible.iterrows():
    elig_dt = datetime.strptime(str(prod["rental_eligible_date"]), "%Y-%m-%d")
    demand  = categories[categories["category_id"]==prod["category_id"]].iloc[0]["rental_demand_tier"]
    stbl    = get_seasonal_table(int(prod["category_id"]), demand)
    price   = prod["original_retail_price"]

    n_rent = np.random.choice([2,3,4,5,6], p=[0.15,0.30,0.30,0.18,0.07])
    cur    = elig_dt + timedelta(days=int(np.random.uniform(1,30)))

    for _ in range(n_rent):
        if cur >= PROG_END: break
        month = cur.month
        if np.random.random() > min(0.98, max(0.55, 0.85 * stbl[month])):
            cur += timedelta(days=int(np.random.uniform(14,30))); continue
        if demand == "high":
            dur = int(np.random.choice([7,14,21,30], p=[0.25,0.35,0.25,0.15]))
        elif demand == "medium":
            dur = int(np.random.choice([7,14,30,45], p=[0.20,0.35,0.30,0.15]))
        else:
            dur = int(np.random.choice([14,30,45,60], p=[0.20,0.35,0.30,0.15]))
        if month in (11,12) and demand == "high":
            dur = int(np.random.choice([7,14,21], p=[0.40,0.35,0.25]))
        end_dt = cur + timedelta(days=dur)
        pm = "pct_of_retail" if (price>500 and np.random.random()<0.65) else \
             ("flat_rate"    if (price<200 and np.random.random()<0.72) else \
             ("flat_rate"    if np.random.random()<0.58 else "pct_of_retail"))
        rule     = pricing[pricing["pricing_model"]==pm].sample(1).iloc[0]
        base_rev = round(rule["base_daily_rate"]*dur, 2) if pm=="flat_rate" \
                   else round(rule["pct_of_retail_daily"]*price*dur, 2)
        if month in (11,12): base_rev = round(base_rev*np.random.uniform(1.05,1.15),2)
        is_late  = np.random.random() < np.random.uniform(0.10,0.15)
        late_d   = int(np.random.uniform(1,8)) if is_late else 0
        late_fee = round(rule["late_fee_per_day"]*late_d, 2) if is_late else 0.0
        ins_fee  = round(base_rev*rule["insurance_fee_pct"], 2)
        total    = round(base_rev+late_fee+ins_fee, 2)
        op_cost  = round(base_rev*round(np.random.uniform(0.15,0.25),3), 2)
        net_rev  = round(total-op_cost, 2)
        no_ret   = np.random.random() < 0.055
        dbr      = no_ret and (np.random.random() < 0.50)
        exp_ret  = end_dt + timedelta(days=late_d)
        act_ret  = exp_ret if not no_ret else None
        rentals_list.append({
            "rental_id":rid, "product_id":int(prod["product_id"]), "customer_id":next_customer(month=month),
            "pricing_rule_id":int(rule["rule_id"]), "rental_start_date":cur.date(), "rental_end_date":end_dt.date(),
            "expected_return_date":exp_ret.date(), "actual_return_date":act_ret.date() if act_ret else None,
            "rental_duration_days":dur, "base_rental_revenue":base_rev, "late_fee":late_fee, "insurance_fee":ins_fee,
            "total_rental_revenue":total, "operational_cost":op_cost, "net_rental_revenue":net_rev,
            "is_no_return":int(no_ret), "is_damaged_beyond_repair":int(dbr), "is_late":int(is_late),
        })
        if not no_ret:
            returns_list.append({"rental_id":rid, "product_id":int(prod["product_id"]),
                "condition_on_return":np.random.choice(["excellent","good","fair","damaged"],p=[0.30,0.45,0.18,0.07]),
                "damage_fee":round(np.random.uniform(0,50),2) if np.random.random()<0.15 else 0.0, "return_note":""})
        events_list.append({"event_id":rid, "product_id":int(prod["product_id"]),
            "event_type":"rental_start", "event_date":cur.date(), "notes":f"rental_id={rid}"})
        rid += 1
        cur = (act_ret or exp_ret) + timedelta(days=int(np.random.uniform(5,15)))

rentals = pd.DataFrame(rentals_list)
returns = pd.DataFrame(returns_list)
events  = pd.DataFrame(events_list)
save("rentals", rentals)
save("return_conditions", returns)
save("inventory_events", events)


  rentals: 1,466 rows
  return_conditions: 1,361 rows
  inventory_events: 1,466 rows


## 8 · Rental vs Discount Comparison

In [10]:
def get_discount(months_unsold, dep_class):
    tiers = {
        "fast":     [(18,.40),(24,.50),(30,.60),(36,.70),(999,.75)],
        "standard": [(18,.30),(24,.40),(30,.50),(36,.60),(999,.65)],
        "slow":     [(18,.25),(24,.30),(30,.40),(36,.50),(999,.55)],
    }
    for thr, pct in tiers[dep_class]:
        if months_unsold <= thr: return pct
    return 0.55

comparison_list = []
for _, prod in products.iterrows():
    pid    = int(prod["product_id"])
    listed = datetime.strptime(str(prod["listed_date"]), "%Y-%m-%d")
    elig   = datetime.strptime(str(prod["rental_eligible_date"]), "%Y-%m-%d")
    months_unsold = (PROG_END - listed).days / 30.44
    cat_row   = categories[categories["category_id"]==prod["category_id"]].iloc[0]
    disc_pct  = get_discount(months_unsold, cat_row["depreciation_class"])
    disc_price = round(prod["original_retail_price"]*(1-disc_pct), 2)
    prod_r = rentals[rentals["product_id"]==pid]
    n_rents = len(prod_r)
    if n_rents > 0:
        net_rev = round(prod_r.iloc[:-1]["net_rental_revenue"].sum(), 2) \
                  if prod_r.iloc[-1]["is_damaged_beyond_repair"] \
                  else round(prod_r["net_rental_revenue"].sum(), 2)
    else:
        net_rev = 0.0
    gross_rev = prod_r["total_rental_revenue"].sum() if n_rents > 0 else 0.0
    op_cost   = prod_r["operational_cost"].sum()      if n_rents > 0 else 0.0
    avg_dur   = prod_r["rental_duration_days"].mean() if n_rents > 0 else 0
    months_on = round(n_rents*avg_dur/30.44, 2)       if n_rents > 0 else 0
    ratio = round(net_rev/disc_price, 4) if disc_price > 0 else 0.0
    comparison_list.append({
        "product_id":                  pid,
        "original_retail_price":       prod["original_retail_price"],
        "months_at_enrollment":        round((elig-listed).days/30.44, 1),
        "months_unsold_at_comparison": round(months_unsold, 1),
        "discount_pct":                disc_pct,
        "hypothetical_discount_price": disc_price,
        "total_gross_rental_revenue":  round(gross_rev, 2),
        "total_operational_cost":      round(op_cost, 2),
        "total_net_rental_revenue":    round(net_rev, 2),
        "n_rentals":                   n_rents,
        "months_on_rental":            months_on,
        "rental_vs_discount_ratio":    ratio,
        "is_rental_more_profitable":   int(ratio > 1.0),
    })
comparison = pd.DataFrame(comparison_list)
save("rental_revenue_vs_discount", comparison)


  rental_revenue_vs_discount: 564 rows


## 9 · Summary

In [11]:
win_rate  = comparison["is_rental_more_profitable"].mean() * 100
avg_ratio = comparison["rental_vs_discount_ratio"].mean()
avg_rents = rentals.groupby("product_id").size().mean()
no_ret    = rentals["is_no_return"].mean() * 100
dbr_rate  = rentals["is_damaged_beyond_repair"].mean() * 100
late_rate = rentals["is_late"].mean() * 100

print("=" * 50)
print("RUN 2 SUMMARY (seed=99)")
print("=" * 50)
print(f"Products:              {len(products)}")
print(f"Customers:             {len(customers):,}")
print(f"Rentals:               {len(rentals):,}")
print(f"Returns:               {len(returns):,}")
print(f"Date range:            {rentals['rental_start_date'].min()} to {rentals['rental_start_date'].max()}")
print(f"Avg rentals/product:   {avg_rents:.1f}")
print(f"No-return rate:        {no_ret:.1f}%")
print(f"Damaged beyond repair: {dbr_rate:.1f}%")
print(f"Late return rate:      {late_rate:.1f}%")
print(f"Rental win rate:       {win_rate:.1f}%")
print(f"Avg ratio:             {avg_ratio:.2f}x")
print("=" * 50)
print("MySQL updated. Go to Power BI → click Refresh.")


RUN 2 SUMMARY (seed=99)
Products:              564
Customers:             2,200
Rentals:               1,466
Returns:               1,361
Date range:            2021-11-24 to 2025-03-30
Avg rentals/product:   2.8
No-return rate:        7.2%
Damaged beyond repair: 3.0%
Late return rate:      11.6%
Rental win rate:       65.2%
Avg ratio:             2.96x
MySQL updated. Go to Power BI → click Refresh.


## 10 · MySQL Analytical Views

In [12]:
views = {
    "v_rental_eligible": (
        "SELECT p.product_id, p.product_name, p.brand, p.original_retail_price, "
        "p.current_depreciated_value, p.listed_date, p.rental_eligible_date, "
        "p.condition_grade, c.category_name, c.depreciation_class, c.rental_demand_tier "
        "FROM products p "
        "JOIN categories c ON p.category_id = c.category_id "
        "WHERE p.rental_eligible_date <= CURDATE()"
    ),
    "v_rental_history": (
        "SELECT r.rental_id, r.product_id, r.customer_id, r.rental_start_date, "
        "r.rental_end_date, r.actual_return_date, r.rental_duration_days, "
        "r.total_rental_revenue, r.net_rental_revenue, r.operational_cost, "
        "r.is_late, r.is_no_return, r.is_damaged_beyond_repair, "
        "p.product_name, p.brand, c.category_name, cu.customer_segment, cu.city, cu.country "
        "FROM rentals r "
        "JOIN products p  ON r.product_id  = p.product_id "
        "JOIN categories c ON p.category_id = c.category_id "
        "JOIN customers cu ON r.customer_id  = cu.customer_id"
    ),
    "v_inventory_aging": (
        "SELECT p.product_id, p.product_name, p.brand, p.listed_date, p.rental_eligible_date, "
        "DATEDIFF(CURDATE(), p.listed_date) AS days_on_shelf, "
        "p.original_retail_price, p.current_depreciated_value, "
        "c.category_name, c.depreciation_class "
        "FROM products p "
        "JOIN categories c ON p.category_id = c.category_id"
    ),
    "v_revenue_comparison": (
        "SELECT rv.product_id, p.product_name, p.brand, "
        "c.category_name, c.depreciation_class, "
        "rv.months_unsold_at_comparison, rv.discount_pct, rv.hypothetical_discount_price, "
        "rv.total_gross_rental_revenue, rv.total_operational_cost, rv.total_net_rental_revenue, "
        "rv.rental_vs_discount_ratio, rv.is_rental_more_profitable, rv.n_rentals, rv.months_on_rental "
        "FROM rental_revenue_vs_discount rv "
        "JOIN products p  ON rv.product_id  = p.product_id "
        "JOIN categories c ON p.category_id  = c.category_id"
    ),
}

with engine.begin() as conn:
    for name, select_sql in views.items():
        conn.execute(text(f"DROP VIEW IF EXISTS `{name}`"))
        conn.execute(text(f"CREATE VIEW `{name}` AS {select_sql}"))

with engine.connect() as conn:
    for name in views:
        n = conn.execute(text(f"SELECT COUNT(*) FROM `{name}`")).scalar()
        print(f"  {name}: {n:,} rows")

for name, select_sql in views.items():
    path = SQL_DIR + "/" + name + ".sql"
    with open(path, "w") as f:
        f.write("DROP VIEW IF EXISTS `" + name + "`;\n")
        f.write("CREATE VIEW `" + name + "` AS " + select_sql + ";\n")
print("Views created. DDL saved to " + SQL_DIR + "/")


  v_rental_eligible: 564 rows
  v_rental_history: 1,466 rows
  v_inventory_aging: 564 rows
  v_revenue_comparison: 564 rows
Views created. DDL saved to ../data/sql/


## 11 · Tableau Export

In [13]:
tableau_data = (
    rentals
    .merge(products[["product_id","category_id","product_name","brand",
                      "original_retail_price","current_depreciated_value",
                      "listed_date","rental_eligible_date","condition_grade"]], on="product_id")
    .merge(categories[["category_id","category_name","avg_depreciation_rate","rental_demand_tier"]], on="category_id")
    .merge(customers[["customer_id","first_name","last_name","city","country","customer_segment"]], on="customer_id")
    .merge(pricing[["rule_id","pricing_model","duration_model","experiment_group"]],
           left_on="pricing_rule_id", right_on="rule_id", how="left")
    .merge(returns[["rental_id","condition_on_return","damage_fee"]], on="rental_id", how="left")
    .merge(comparison[["product_id","hypothetical_discount_price","total_net_rental_revenue",
                        "total_operational_cost","rental_vs_discount_ratio","is_rental_more_profitable"]],
           on="product_id", how="left")
)
tableau_data["customer_name"] = tableau_data["first_name"] + " " + tableau_data["last_name"]
tableau_data.drop(columns=["first_name","last_name","rule_id"], inplace=True, errors="ignore")
tableau_data.to_csv(f"{TABLEAU_DIR}/rental_analysis_full.csv", index=False)
print(f"rental_analysis_full.csv: {len(tableau_data):,} rows, {len(tableau_data.columns)} columns")

profitability = comparison.merge(
    products[["product_id","product_name","brand","category_id"]], on="product_id"
).merge(categories[["category_id","category_name"]], on="category_id")
profitability.to_csv(f"{TABLEAU_DIR}/product_profitability.csv", index=False)
print(f"product_profitability.csv: {len(profitability)} rows")
print(f"Tableau files saved to {TABLEAU_DIR}/")


rental_analysis_full.csv: 1,466 rows, 43 columns
product_profitability.csv: 564 rows
Tableau files saved to ../data/tableau/


---
## Done

Run top to bottom. Every section prints row counts as it goes.
Now go to Power BI → click **Atualizar** (Refresh) → dashboard updates automatically.